# Dirichlet-main learning curves

This notebook scans every `alpha/dataset` category under `experiment/01_dirichlet_main` and creates one three-panel PDF per category. The panels compare the four aggregation methods under **Correct**, **Medium**, and **Mismatch** rank allocation.

In [ ]:
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

RANK_SCENARIOS = ["correct", "medium", "mismatch"]
PANEL_TITLES = {
    "correct": "(a) Correct",
    "medium": "(b) Medium",
    "mismatch": "(c) Mismatch",
}
METHOD_STYLES = {
    "rbla_plus": {"label": "RBLA+", "color": "#D55E00", "linestyle": "-",  "linewidth": 2.6, "zorder": 4},
    "rbla":      {"label": "RBLA",  "color": "#0072B2", "linestyle": "-",  "linewidth": 1.9, "zorder": 3},
    "sp":        {"label": "SP",    "color": "#E69F00", "linestyle": "--", "linewidth": 1.9, "zorder": 2},
    "zeropadding": {"label": "Zero-padding", "color": "#009E73", "linestyle": "-.", "linewidth": 1.9, "zorder": 1},
}
DATASET_LABELS = {
    "fmnist": "Fashion-MNIST",
    "kmnist": "KMNIST",
    "mnist": "MNIST",
    "qmnist": "QMNIST",
}

In [ ]:
def locate_repo_root() -> Path:
    """Locate the repository whether Jupyter starts here or in an ancestor."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        if (base / "rbla+" / "experiment" / "01_dirichlet_main").is_dir():
            return base
    raise FileNotFoundError("Could not locate rbla+/experiment/01_dirichlet_main")

REPO_ROOT = locate_repo_root()
EXPERIMENT_ROOT = REPO_ROOT / "rbla+" / "experiment" / "01_dirichlet_main"
OUTPUT_ROOT = REPO_ROOT / "rbla+" / "figures" / "dirichlet_main" / "pdf"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Input : {EXPERIMENT_ROOT}")
print(f"Output: {OUTPUT_ROOT}")

In [ ]:
def read_training_csv(path: Path) -> pd.DataFrame:
    """Read a training CSV after its Config preamble."""
    lines = path.read_text(encoding="utf-8-sig").splitlines(keepends=True)
    header_index = next(
        (i for i, line in enumerate(lines) if line.strip().startswith("round,accuracy,")),
        None,
    )
    if header_index is None:
        raise ValueError(f"Training table header not found in {path}")

    frame = pd.read_csv(StringIO("".join(lines[header_index:])))
    required = {"round", "accuracy"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing columns {sorted(missing)} in {path}")

    frame = frame.loc[:, ["round", "accuracy"]].dropna().copy()
    frame["round"] = pd.to_numeric(frame["round"], errors="raise")
    frame["accuracy"] = pd.to_numeric(frame["accuracy"], errors="raise")
    return frame.sort_values("round")


def method_from_filename(path: Path) -> str:
    name = path.name.lower()
    for method in METHOD_STYLES:
        if f"_{method}_-train-" in name:
            return method
    raise ValueError(f"Cannot identify method from filename: {path.name}")


def collect_category(category_dir: Path) -> dict[str, dict[str, Path]]:
    result = {}
    for scenario in RANK_SCENARIOS:
        scenario_dir = category_dir / scenario
        files = sorted(scenario_dir.glob("*.csv"))
        by_method = {}
        for path in files:
            method = method_from_filename(path)
            if method in by_method:
                raise ValueError(f"Duplicate {method} runs in {scenario_dir}")
            by_method[method] = path

        missing = set(METHOD_STYLES).difference(by_method)
        if missing:
            raise ValueError(f"Missing methods {sorted(missing)} in {scenario_dir}")
        result[scenario] = by_method
    return result

In [ ]:
def alpha_label(alpha_dir_name: str) -> str:
    return alpha_dir_name.removeprefix("alpha_").replace("_", ".")


def plot_category(category_dir: Path) -> Path:
    alpha_name = category_dir.parent.name
    dataset_name = category_dir.name.lower()
    runs = collect_category(category_dir)

    fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.55), sharex=True, sharey=True)
    legend_handles = []

    for axis, scenario in zip(axes, RANK_SCENARIOS):
        for method, style in METHOD_STYLES.items():
            frame = read_training_csv(runs[scenario][method])
            line, = axis.plot(
                frame["round"],
                frame["accuracy"] * 100.0,
                label=style["label"],
                color=style["color"],
                linestyle=style["linestyle"],
                linewidth=style["linewidth"],
                zorder=style["zorder"],
            )
            if scenario == RANK_SCENARIOS[0]:
                legend_handles.append(line)

        axis.set_title(PANEL_TITLES[scenario], pad=7)
        axis.set_xlabel("Communication Round")
        axis.set_xlim(0, 100)
        axis.set_ylim(0, 100)
        axis.set_xticks(range(0, 101, 20))
        axis.set_yticks(range(0, 101, 20))
        axis.grid(True, linewidth=0.6, alpha=0.55)
        axis.set_axisbelow(True)

    axes[0].set_ylabel("Test Accuracy (%)")
    dataset_label = DATASET_LABELS.get(dataset_name, dataset_name.upper())
    fig.suptitle(
        rf"{dataset_label}, Dirichlet $\alpha={alpha_label(alpha_name)}$",
        fontsize=13,
        y=1.08,
    )
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=len(METHOD_STYLES),
        frameon=False,
        handlelength=3.0,
        columnspacing=1.8,
    )
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.16, top=0.76, wspace=0.08)

    output_dir = OUTPUT_ROOT / alpha_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{dataset_name}_learning_curve.pdf"
    fig.savefig(
        output_path,
        format="pdf",
        bbox_inches="tight",
        metadata={
            "Title": f"{dataset_label} Dirichlet alpha={alpha_label(alpha_name)} learning curves",
            "Subject": "Comparison of RBLA+, RBLA, SP, and Zero-padding",
        },
    )
    plt.close(fig)
    return output_path


In [ ]:
category_dirs = sorted(
    dataset_dir
    for alpha_dir in EXPERIMENT_ROOT.glob("alpha_*")
    if alpha_dir.is_dir()
    for dataset_dir in alpha_dir.iterdir()
    if dataset_dir.is_dir() and all((dataset_dir / name).is_dir() for name in RANK_SCENARIOS)
)

if not category_dirs:
    raise RuntimeError(f"No alpha/dataset categories found under {EXPERIMENT_ROOT}")

generated_pdfs = [plot_category(category_dir) for category_dir in category_dirs]
print(f"Generated {len(generated_pdfs)} PDF files:")
for path in generated_pdfs:
    print(f"  {path.relative_to(REPO_ROOT)}")